
# 🎾 Tennis Fatigue Analysis: Cloud Pipeline

## Overview
This notebook implements a high-performance, cloud-based pipeline for processing multi-modal sensor data (EMG, IMU, and Vitals) captured from tennis players. It is designed to overcome the processing limitations of local machines by leveraging **parallel execution** to analyze hours of high-frequency data in minutes.

## Key Objectives
1.  **Massive Parallelization:** Uses `multiprocessing` to process multiple match sessions simultaneously, significantly reducing analysis time.
2.  **Automated Feature Extraction:** Extracts biomechanical (swing speed, impact shock) and neuromuscular (EMG RMS, Median Frequency) features for every swing.
3.  **Fatigue & Efficiency Analysis:** Calculates the **Neuromechanical Efficiency Ratio (NMER)** to identify when a player's swing efficiency begins to degrade—a key predictor of overuse injury.
4.  **AI-Ready Dataset Creation:** Aggregates individual session data into a single `MASTER_TRAINING_DATA.csv` for training future Machine Learning models.

## Pipeline Workflow
1.  **Setup:** Mounts Google Drive and imports custom analysis scripts (`emg_util`, `runner`, etc.).
2.  **Configuration:** Defines analysis parameters (thresholds, sample rates) directly in the notebook for rapid experimentation.
3.  **Discovery:** Scans the input directory for raw CSV logs generated by the ESP32/Python capture system.
4.  **Execution:** Spawns a pool of worker processes to analyze each session in parallel.
5.  **Visualization:** Aggregates results and plots the "Cost of Swing" efficiency trend across all datasets.

---
**Input:** Raw CSV logs (`emg_log_*.csv`, `imu_log_*.csv`) stored in Google Drive.
**Output:** Processed feature CSVs, fatigue summary plots, and a consolidated training dataset.

##  Step1: Setup & Imports

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Add your script folder to Python Path (So we can import your scripts)
import sys
import os
# CHANGE THIS PATH to where your python scripts live in Drive
SCRIPT_DIR = '/content/drive/MyDrive/Tennis_Project/python_scripts'
sys.path.append(SCRIPT_DIR)

# 3. Install dependencies
!pip install pyyaml scipy pandas matplotlib

# 4. Import your modules
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import emg_util_yaml_runner as runner
import capture_loader as loader
import pandas as pd
import matplotlib.pyplot as plt

Fri Feb  7 07:19:14 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   28C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step2: Configuration (The "Cloud YAML")


In [ ]:
# Instead of loading a file, we define the config dictionary right here.
# This makes it super easy to change parameters and re-run.

CLOUD_CONFIG = {
    'session': {
        'emg_glob': 'emg_log_*.csv',
        'imu_glob': 'imu_log_*.csv',
        'vitals_path': 'vitals_log.csv',
        'emg_fs_hz': 2000,
        'acc_fsr_default_g': 2,
        'gyro_fsr_default_dps': 250,
        'use_physical_units': True
    },
    'features': {
        'mode': 'both', # 'bout' or 'both'
        'zc_thr': 0.01,
        'ssc_thr': 0.01
    },
    'swing_detection': {
        'use_gyro': True,
        'z_thr': 5.0,
        'min_consec': 3,
        'min_gap_ms': 300.0,
        'pre_ms': 250.0,
        'post_ms': 450.0
    },
    'baseline': {
        'warmup_skip_min': 5.0,
        'baseline_swings': 20
    },
    'plots': {
        'enable': True,
        'full_duration': True,
        'do_fatigue': True
    }
}

# Define Paths
# CHANGE THIS to where your raw CSV logs are
LOG_DIR = '/content/drive/MyDrive/Tennis_Project/logs'
# CHANGE THIS to where you want the results
OUT_DIR = '/content/drive/MyDrive/Tennis_Project/processed_output'

##Step3: Discovery & Parallel Execution


In [ ]:
# 1. Discover Sessions
print(f"Scanning {LOG_DIR}...")
sessions = loader.discover_sessions(
    LOG_DIR,
    emg_glob=CLOUD_CONFIG['session']['emg_glob'],
    imu_glob=CLOUD_CONFIG['session']['imu_glob'],
    vitals_name=CLOUD_CONFIG['session']['vitals_path']
)

session_ids = sorted(sessions.keys())
print(f"Found {len(session_ids)} sessions: {session_ids}")

# 2. Wrapper for Parallel Mapper
def process_wrapper(session_id):
    """Helper to unpack arguments for the runner"""
    files = sessions[session_id]
    try:
        # Call the function we refactored earlier!
        out_path = runner.analyze_single_session(session_id, files, CLOUD_CONFIG, OUT_DIR)
        return (session_id, "SUCCESS", out_path)
    except Exception as e:
        return (session_id, "FAILED", str(e))

# 3. Run in Parallel
# Colab usually gives 2 cores (Standard) or more (Pro).
# Using ProcessPoolExecutor to max out CPU.

if __name__ == '__main__':
    num_workers = multiprocessing.cpu_count()
    print(f"Starting parallel processing on {num_workers} cores...")

    results = []
    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        # Submit all jobs
        futures = executor.map(process_wrapper, session_ids)

        # Collect results
        for res in futures:
            results.append(res)
            print(f"Finished: {res[0]} -> {res[1]}")

print("\n--- Processing Complete ---")

##Step4: Aggregate Results & AI Prep


In [ ]:
# Now we read all the 'swing_features.csv' files generated above
# and create one MASTER dataset for your AI training.

all_swings = []

for session_id in session_ids:
    csv_path = os.path.join(OUT_DIR, session_id, 'swing_features.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        df['session_id'] = session_id # Add ID so we know who/when this was
        all_swings.append(df)

if all_swings:
    MASTER_DF = pd.concat(all_swings, ignore_index=True)
    print(f"Loaded {len(MASTER_DF)} total swings.")

    # Save this Master Dataset for your AI Training
    save_path = os.path.join(OUT_DIR, 'MASTER_TRAINING_DATA.csv')
    MASTER_DF.to_csv(save_path, index=False)
    print(f"Saved Master Training Data to: {save_path}")

    # Quick Check: Plot Efficiency Ratio (RMS / Speed)
    plt.figure(figsize=(10, 6))
    plt.scatter(MASTER_DF['t_start_ms'], MASTER_DF['rms'] / MASTER_DF['imu_peak_mag'], alpha=0.5)
    plt.xlabel('Time (ms)')
    plt.ylabel('Cost Ratio (RMS / Gyro)')
    plt.title('Cost of Swing over Time (All Sessions)')
    plt.show()
else:
    print("No swing features found.")